# Gold Dataset Creation & Subdataset Cleaning Pipeline

This notebook implements an end-to-end pipeline to:
1.  **Load** the unified arXiv dataset.
2.  **Define** three subdataset selection strategies (Top N, Water-filling, Quartile).
3.  **Create Gold Datasets**: For each strategy, iteratively select articles, download their LaTeX source, and extract text chunks containing citations until 20 valid articles are processed.
4.  **Clean Subdatasets**: Generate the final subdatasets with the 20 Gold articles excluded.
5.  **Save** all outputs in JSON format.

In [ ]:
import json
import os
import re
import random
import tarfile
import gzip
import shutil
import time
import requests
import io
from typing import List, Dict, Set, Any, Tuple, Optional
from collections import defaultdict

## 1. Configuration & Helper Functions

We define paths and regex patterns for parsing LaTeX citations. The `clean_text` function removes LaTeX formatting to leave readable text.

In [ ]:
# Configuration
DATA_PATH = 'data/processed/repaired_unified_articles.json'
OUTPUT_DIR = 'data/gold_datasets/'
TEMP_DIR = 'temp_latex_downloads/'
TARGET_GOLD_SIZE = 20
CONTEXT_WINDOW = 300  # Characters before/after citation to extract

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

if not os.path.exists(TEMP_DIR):
    os.makedirs(TEMP_DIR)

# Regex for finding citations: \cite{...}, \citep{...}, etc.
# Captures the whole command and the inner content.
CITE_PATTERN = re.compile(r'(\\(cite|citep|citet|bibcite)\{([^}]+)\})')

def clean_latex_text(text: str) -> str:
    """
    Simple cleaner to remove common LaTeX commands and excess whitespace 
    from a text chunk to make it more readable.
    """
    # Remove comments
    text = re.sub(r'%.*', '', text)
    # Replace standard LaTeX commands like \textbf{word} with word
    text = re.sub(r'\\(textbf|textit|emph|section|subsection)\{([^}]+)\}', r'\2', text)
    # Remove other backslash commands
    # text = re.sub(r'\\[a-zA-Z]+', '', text) # Aggressive removal (optional)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

## 2. Data Loading

Load the unified dataset containing article metadata and references.

In [ ]:
def load_data(filepath: str) -> List[Dict]:
    """Loads the unified articles JSON dataset."""
    print(f"Loading dataset from {filepath}...")
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Handle both dict with 'articles' key or list format
    if isinstance(data, dict) and 'articles' in data:
        articles = data['articles']
    else:
        articles = data
        
    print(f"Loaded {len(articles)} articles.")
    return articles

articles_data = load_data(DATA_PATH)

## 3. Article Download & Processing

These functions handle the complexity of retrieving source files from arXiv:
1.  `download_source`: Fetches the `.tar.gz` or `.pdf` from arXiv.
2.  `extract_tex_content`: Unpacks the source and finds `.tex` files.
3.  `extract_citation_chunks`: Scans the `.tex` content for citations and extracts the surrounding context.

In [ ]:
def download_source(arxiv_id: str, save_dir: str) -> Optional[str]:
    """
    Downloads the source file for a given arXiv ID.
    Returns the path to the downloaded file if successful, None otherwise.
    """
    # Ensure ID is clean (remove version numbers if necessary, though arXiv usually redirects)
    url = f"https://arxiv.org/e-print/{arxiv_id}"
    save_path = os.path.join(save_dir, f"{arxiv_id}.tar.gz")
    
    try:
        response = requests.get(url, stream=True, timeout=15)
        if response.status_code == 200:
            # Check content type to ensure it's not a PDF (some old papers only have PDF)
            content_type = response.headers.get('content-type', '')
            if 'pdf' in content_type:
                # print(f"Skipping {arxiv_id}: Source unavailable (PDF only).")
                return None
                
            with open(save_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            return save_path
        else:
            # print(f"Failed to download {arxiv_id}: Status {response.status_code}")
            return None
    except Exception as e:
        print(f"Error downloading {arxiv_id}: {e}")
        return None

def extract_tex_content(file_path: str) -> str:
    """
    Extracts text content from a downloaded arXiv source file.
    Handles .tar.gz archives (multiple files) and single .gz files.
    Merges all found .tex files into one string.
    """
    full_latex_content = ""
    
    try:
        # Attempt to open as tar file
        if tarfile.is_tarfile(file_path):
            with tarfile.open(file_path, 'r:*') as tar:
                for member in tar.getmembers():
                    if member.name.endswith('.tex'):
                        try:
                            f = tar.extractfile(member)
                            content = f.read().decode('utf-8', errors='ignore')
                            full_latex_content += "\n% FILE START: " + member.name + "\n"
                            full_latex_content += content
                        except Exception:
                            continue
        else:
            # Attempt to open as single gzipped file
            try:
                with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
                    full_latex_content = f.read()
            except Exception:
                pass
                
    except Exception as e:
        print(f"Error extraction {file_path}: {e}")
        
    return full_latex_content

def extract_citation_chunks(latex_text: str) -> List[str]:
    """
    Finds all occurrences of \cite{...} and extracts the surrounding context.
    Returns a list of strings (chunks).
    """
    chunks = []
    if not latex_text:
        return chunks
        
    for match in CITE_PATTERN.finditer(latex_text):
        start_idx = match.start()
        end_idx = match.end()
        
        # Calculate context window
        context_start = max(0, start_idx - CONTEXT_WINDOW)
        context_end = min(len(latex_text), end_idx + CONTEXT_WINDOW)
        
        raw_chunk = latex_text[context_start:context_end]
        
        # Optional: Clean up the chunk (remove excessive LaTeX formatting)
        clean_chunk = clean_latex_text(raw_chunk)
        
        if len(clean_chunk) > 50: # Filter out tiny scraps
            chunks.append(clean_chunk)
            
    return chunks

## 4. Strategy Implementations

We implement the three required subdataset creation strategies to generate candidate lists.

1.  **Most Cited (Top N)**: Simple sort by citation count.
2.  **Stratified (Water Filling)**: Balanced selection across categories.
3.  **Quartile**: Selection based on citation quartiles (e.g., Q1 top papers).

In [ ]:
def get_candidates_most_cited(articles: List[Dict], top_n: int = 1000) -> List[Dict]:
    """Returns top N articles sorted by number of references (or citations if available)."""
    # Sorting by number of OUTGOING references as a proxy if citation count isn't in metadata.
    # If you have an 'citations_count' field, use that instead.
    # Assuming 'refs' field exists from preprocessing.
    sorted_articles = sorted(articles, key=lambda x: len(x.get('refs', [])), reverse=True)
    return sorted_articles[:top_n]

def get_candidates_stratified(articles: List[Dict], total_k: int = 1000) -> List[Dict]:
    """Returns K articles distributed across categories (Water Filling)."""
    # 1. Group by category
    cat_map = defaultdict(list)
    for art in articles:
        cats = art.get('categories', '').split()
        if cats:
            primary_cat = cats[0]
            cat_map[primary_cat].append(art)
            
    # 2. Calculate target per category
    n_cats = len(cat_map)
    if n_cats == 0: return []
    target_per_cat = max(1, total_k // n_cats)
    
    selected = []
    # 3. Select top cited from each category
    for cat, arts in cat_map.items():
        # Sort by refs count
        sorted_arts = sorted(arts, key=lambda x: len(x.get('refs', [])), reverse=True)
        selected.extend(sorted_arts[:target_per_cat])
        
    return selected[:total_k]

def get_candidates_quartile(articles: List[Dict], quartile: int = 1, total_n: int = 1000) -> List[Dict]:
    """Returns articles from a specific citation quartile (1=Top 25%)."""
    # Sort all by ref count
    sorted_articles = sorted(articles, key=lambda x: len(x.get('refs', [])), reverse=True)
    n = len(sorted_articles)
    
    q_size = n // 4
    start_idx = (quartile - 1) * q_size
    end_idx = start_idx + q_size
    
    # Get the slice for the quartile
    quartile_pool = sorted_articles[start_idx:end_idx]
    
    # Return top N from this quartile (or random, but top is consistent)
    return quartile_pool[:total_n]

## 5. Main Processing Pipeline

The `create_gold_dataset` function is the engine. It:
1. Takes a candidate list.
2. Iterates through candidates, trying to download and parse them.
3. Stops when 20 valid articles (with citations found) are collected.
4. Returns the Gold List and the IDs to be removed from the subdataset.

In [ ]:
def create_gold_dataset(candidates: List[Dict], target_size: int = 20) -> Tuple[List[Dict], List[str]]:
    """
    Tries to create a gold dataset of `target_size` from the candidates.
    Returns:
        gold_data: List of dicts {id: ..., chunks: ...}
        gold_ids: List of IDs to remove from the subdataset
    """
    gold_data = []
    gold_ids = []
    processed_count = 0
    
    print(f"Starting Gold Dataset creation (Target: {target_size})...")
    
    for article in candidates:
        if len(gold_data) >= target_size:
            break
            
        aid = article['id']
        processed_count += 1
        print(f"[{len(gold_data)}/{target_size}] Processing {aid}...")
        
        # 1. Download Source
        source_path = download_source(aid, TEMP_DIR)
        if not source_path:
            continue
            
        # 2. Extract LaTeX content
        latex_content = extract_tex_content(source_path)
        
        # 3. Extract Chunks
        chunks = extract_citation_chunks(latex_content)
        
        # 4. Validate
        if len(chunks) > 0:
            gold_entry = {
                "id": aid,
                "chunks": chunks
            }
            gold_data.append(gold_entry)
            gold_ids.append(aid)
            print(f" -> Success: Found {len(chunks)} citation chunks.")
        else:
            print(" -> Skipped: No readable citations found in source.")
            
        # Cleanup temp file to save space
        if os.path.exists(source_path):
            os.remove(source_path)
            
        # Be polite to arXiv API
        time.sleep(2)
        
    print(f"Finished. Collected {len(gold_data)} articles.")
    return gold_data, gold_ids

In [ ]:
# ==========================================
# EXECUTION PHASE
# ==========================================

strategies = {
    "most_cited": lambda data: get_candidates_most_cited(data, top_n=200),
    "stratified": lambda data: get_candidates_stratified(data, total_k=200),
    "quartile_q1": lambda data: get_candidates_quartile(data, quartile=1, total_n=200)
}

for strat_name, strat_func in strategies.items():
    print(f"\n=== Processing Strategy: {strat_name} ===")
    
    # 1. Generate Candidates
    candidates = strat_func(articles_data)
    print(f"Generated {len(candidates)} candidates.")
    
    # 2. Create Gold Dataset (Download & Extract)
    gold_dataset, gold_ids_to_remove = create_gold_dataset(candidates, target_size=TARGET_GOLD_SIZE)
    
    # 3. Save Gold Dataset
    gold_output_path = os.path.join(OUTPUT_DIR, f"gold_dataset_{strat_name}.json")
    with open(gold_output_path, 'w', encoding='utf-8') as f:
        json.dump(gold_dataset, f, indent=2)
    print(f"Saved Gold Dataset to {gold_output_path}")
    
    # 4. Create Clean Subdataset (Original Candidates - Gold Articles)
    # Note: This creates a subdataset of the remaining candidates. 
    # If you want the subdataset to be larger (e.g. 10k), increase the candidate generation limit above.
    clean_candidates = [art for art in candidates if art['id'] not in gold_ids_to_remove]
    
    subdataset_output_path = os.path.join(OUTPUT_DIR, f"subdataset_{strat_name}_clean.json")
    with open(subdataset_output_path, 'w', encoding='utf-8') as f:
        json.dump({"articles": clean_candidates, "count": len(clean_candidates)}, f, indent=2)
    print(f"Saved Clean Subdataset to {subdataset_output_path}")

In [ ]:
# Cleanup Temp Directory
shutil.rmtree(TEMP_DIR, ignore_errors=True)
print("\nAll tasks completed. Temporary files removed.")